# Module 5: Hybrid Methods - GA with Local Search and Memetic Algorithms

## Learning Objectives

By completing this notebook, you will:
1. Understand the synergy between global and local search
2. Implement hybrid GA combining evolutionary and hill-climbing approaches
3. Design memetic algorithms with individual learning
4. Apply Lamarckian and Baldwinian evolution strategies
5. Compare hybrid methods with pure GA and pure local search
6. Optimize the balance between exploration and exploitation

## Prerequisites

- Module 4 completed (GA implementation)
- Understanding of local search algorithms
- Knowledge of exploration vs exploitation trade-off
- DEAP library proficiency

## Estimated Time: 100 minutes

---

## 1. Why Hybrid Algorithms?

### Key Concept: Combine strengths of different search strategies.

**Pure GA Characteristics:**
- Excellent global exploration
- Good at escaping local optima
- Slow fine-tuning of solutions
- Population-based diversity

**Pure Local Search Characteristics:**
- Fast convergence to local optimum
- Excellent exploitation
- Gets stuck in local optima
- Single-solution focused

**Hybrid Approach Benefits:**
1. **Best of both worlds**: Global search + local refinement
2. **Faster convergence**: Local search polishes GA solutions
3. **Better solutions**: Fine-tuning improves quality
4. **Efficiency**: Fewer evaluations to reach good solutions

**Types of Hybrid GAs:**
- **GA + Hill Climbing**: Apply local search after crossover/mutation
- **Memetic Algorithm**: Each individual learns independently
- **Lamarckian**: Local improvements inherited (modify genome)
- **Baldwinian**: Local improvements affect fitness only (no genome change)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random
import copy
import time

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

### 1.1 Load and Prepare Dataset

In [ ]:
# Purpose: Prepare dataset for hybrid algorithm comparison
# Key Concept: Medium-sized dataset for efficient experimentation

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

N_FEATURES = X_train_scaled.shape[1]

print(f"Dataset: {X_train_scaled.shape[0]} train, {X_test_scaled.shape[0]} test")
print(f"Features: {N_FEATURES}")

### 1.2 Setup DEAP Framework

In [ ]:
# Purpose: Initialize DEAP for hybrid algorithms
# Key Concept: Standard setup, will add local search later

# Clean up existing types
if hasattr(creator, "FitnessMax"):
    del creator.FitnessMax
if hasattr(creator, "Individual"):
    del creator.Individual

# Create types
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Fitness function
def evaluate_features(individual, X_data, y_data):
    """Standard fitness evaluation."""
    if sum(individual) == 0:
        return (-999.0,)
    
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, feature_mask]
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_selected, y_data, cv=3, scoring='accuracy')
    
    accuracy = scores.mean()
    parsimony = 0.01 * (sum(individual) / len(individual))
    
    return (accuracy - parsimony,)

print("DEAP framework initialized.")

## 2. Local Search Operators

### Key Concept: Local search explores neighborhood of a solution.

### 2.1 Bit-Flip Hill Climbing

In [ ]:
# Purpose: Implement 1-bit flip hill climbing
# Key Concept: Try flipping each bit, keep if improvement

def bit_flip_hill_climb(individual, evaluate_func, max_iterations=10):
    """
    Hill climbing with 1-bit flips.
    
    Tries flipping each bit individually, accepts improvements.
    
    Parameters
    ----------
    individual : Individual
        Starting solution
    evaluate_func : callable
        Fitness evaluation function
    max_iterations : int
        Maximum hill climbing iterations
    
    Returns
    -------
    improved : Individual
        Locally optimized solution
    n_evaluations : int
        Number of fitness evaluations performed
    """
    improved = copy.deepcopy(individual)
    current_fitness = evaluate_func(improved)[0]
    n_evaluations = 1
    
    for iteration in range(max_iterations):
        improved_this_iter = False
        
        # Try flipping each bit
        for i in range(len(improved)):
            # Flip bit i
            improved[i] = 1 - improved[i]
            
            # Evaluate
            new_fitness = evaluate_func(improved)[0]
            n_evaluations += 1
            
            if new_fitness > current_fitness:
                # Keep improvement
                current_fitness = new_fitness
                improved_this_iter = True
                break  # First improvement strategy
            else:
                # Revert flip
                improved[i] = 1 - improved[i]
        
        # Stop if no improvement
        if not improved_this_iter:
            break
    
    return improved, n_evaluations

# Test hill climbing
test_individual = creator.Individual([random.randint(0, 1) for _ in range(N_FEATURES)])
initial_fitness = evaluate_features(test_individual, X_train_scaled, y_train)[0]

print("Testing hill climbing...")
print(f"Initial fitness: {initial_fitness:.4f}, Features: {sum(test_individual)}")

improved_ind, n_evals = bit_flip_hill_climb(
    test_individual,
    lambda ind: evaluate_features(ind, X_train_scaled, y_train),
    max_iterations=5
)

final_fitness = evaluate_features(improved_ind, X_train_scaled, y_train)[0]
print(f"Final fitness: {final_fitness:.4f}, Features: {sum(improved_ind)}")
print(f"Improvement: {final_fitness - initial_fitness:.4f}")
print(f"Evaluations used: {n_evals}")

### 2.2 Greedy Feature Addition/Removal

In [ ]:
# Purpose: Implement greedy forward/backward feature search
# Key Concept: Add best feature or remove worst feature

def greedy_local_search(individual, evaluate_func, max_steps=5):
    """
    Greedy forward selection and backward elimination.
    
    Alternates between adding most beneficial feature and
    removing least beneficial feature.
    
    Parameters
    ----------
    individual : Individual
        Starting solution
    evaluate_func : callable
        Fitness evaluation function
    max_steps : int
        Maximum number of add/remove operations
    
    Returns
    -------
    improved : Individual
        Locally optimized solution
    n_evaluations : int
        Number of evaluations
    """
    improved = copy.deepcopy(individual)
    current_fitness = evaluate_func(improved)[0]
    n_evaluations = 1
    
    for step in range(max_steps):
        best_improvement = 0
        best_action = None
        
        # Try adding each unselected feature
        for i in range(len(improved)):
            if improved[i] == 0:
                improved[i] = 1
                new_fitness = evaluate_func(improved)[0]
                n_evaluations += 1
                
                improvement = new_fitness - current_fitness
                if improvement > best_improvement:
                    best_improvement = improvement
                    best_action = ('add', i)
                
                improved[i] = 0  # Revert
        
        # Try removing each selected feature
        for i in range(len(improved)):
            if improved[i] == 1:
                improved[i] = 0
                new_fitness = evaluate_func(improved)[0]
                n_evaluations += 1
                
                improvement = new_fitness - current_fitness
                if improvement > best_improvement:
                    best_improvement = improvement
                    best_action = ('remove', i)
                
                improved[i] = 1  # Revert
        
        # Apply best action
        if best_action is not None:
            action_type, idx = best_action
            if action_type == 'add':
                improved[idx] = 1
            else:
                improved[idx] = 0
            current_fitness += best_improvement
        else:
            # No improvement found
            break
    
    return improved, n_evaluations

# Test greedy search
test_individual2 = creator.Individual([random.randint(0, 1) for _ in range(N_FEATURES)])
initial_fitness2 = evaluate_features(test_individual2, X_train_scaled, y_train)[0]

print("\nTesting greedy local search...")
print(f"Initial fitness: {initial_fitness2:.4f}, Features: {sum(test_individual2)}")

improved_ind2, n_evals2 = greedy_local_search(
    test_individual2,
    lambda ind: evaluate_features(ind, X_train_scaled, y_train),
    max_steps=3
)

final_fitness2 = evaluate_features(improved_ind2, X_train_scaled, y_train)[0]
print(f"Final fitness: {final_fitness2:.4f}, Features: {sum(improved_ind2)}")
print(f"Improvement: {final_fitness2 - initial_fitness2:.4f}")
print(f"Evaluations used: {n_evals2}")

## 3. Lamarckian Evolution

### Key Concept: Local improvements are inherited (modify genome).

In [ ]:
# Purpose: Implement Lamarckian hybrid GA
# Key Concept: Apply local search and update individual's genome

def lamarckian_ga(toolbox, population_size=50, n_generations=30,
                 local_search_prob=0.3, local_search_iters=3):
    """
    Genetic algorithm with Lamarckian local search.
    
    After crossover/mutation, apply local search with some probability.
    Improved individuals replace originals (Lamarckian inheritance).
    
    Parameters
    ----------
    toolbox : deap.Toolbox
        DEAP toolbox with registered operators
    population_size : int
        Population size
    n_generations : int
        Number of generations
    local_search_prob : float
        Probability of applying local search to offspring
    local_search_iters : int
        Iterations for local search
    
    Returns
    -------
    population : list
        Final population
    logbook : Logbook
        Evolution statistics
    total_ls_evals : int
        Total local search evaluations
    """
    # Statistics
    stats = tools.Statistics(key=lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("max", np.max)
    
    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + stats.fields
    
    # Initialize population
    population = toolbox.population(n=population_size)
    
    # Evaluate initial population
    fitnesses = map(toolbox.evaluate, population)
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
    
    total_ls_evals = 0
    
    # Evolution
    for gen in range(n_generations):
        # Select
        offspring = toolbox.select(population, len(population))
        offspring = [toolbox.clone(ind) for ind in offspring]
        
        # Crossover
        for i in range(1, len(offspring), 2):
            if random.random() < 0.7:
                offspring[i-1], offspring[i] = toolbox.mate(offspring[i-1], offspring[i])
                del offspring[i-1].fitness.values
                del offspring[i].fitness.values
        
        # Mutation
        for mutant in offspring:
            if random.random() < 0.2:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        
        # Evaluate
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
        
        # Lamarckian local search
        for ind in offspring:
            if random.random() < local_search_prob:
                # Apply local search
                improved, n_evals = bit_flip_hill_climb(
                    ind,
                    toolbox.evaluate,
                    max_iterations=local_search_iters
                )
                total_ls_evals += n_evals
                
                # Replace individual (Lamarckian)
                ind[:] = improved
                ind.fitness.values = toolbox.evaluate(ind)
        
        # Replace population
        population[:] = offspring
        
        # Record statistics
        record = stats.compile(population)
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        
        if (gen + 1) % 10 == 0:
            print(f"Gen {gen+1}: Best={record['max']:.4f}, Avg={record['avg']:.4f}")
    
    return population, logbook, total_ls_evals

print("Lamarckian GA function defined.")

## 4. Baldwinian Evolution

### Key Concept: Local improvements affect fitness but not genome.

In [ ]:
# Purpose: Implement Baldwinian hybrid GA
# Key Concept: Evaluate with local search but don't modify genome

def baldwinian_ga(toolbox, population_size=50, n_generations=30,
                 local_search_prob=0.3, local_search_iters=3):
    """
    Genetic algorithm with Baldwinian local search.
    
    Local search improves fitness evaluation but doesn't change genome.
    This rewards individuals with good learning potential.
    
    Parameters
    ----------
    Same as lamarckian_ga
    
    Returns
    -------
    Same as lamarckian_ga
    """
    stats = tools.Statistics(key=lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("max", np.max)
    
    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + stats.fields
    
    population = toolbox.population(n=population_size)
    
    # Evaluate with Baldwinian learning
    for ind in population:
        base_fitness = toolbox.evaluate(ind)[0]
        
        # Try local search
        if random.random() < local_search_prob:
            improved, _ = bit_flip_hill_climb(ind, toolbox.evaluate, local_search_iters)
            learned_fitness = toolbox.evaluate(improved)[0]
            # Use improved fitness but don't change genome
            ind.fitness.values = (learned_fitness,)
        else:
            ind.fitness.values = (base_fitness,)
    
    total_ls_evals = 0
    
    for gen in range(n_generations):
        offspring = toolbox.select(population, len(population))
        offspring = [toolbox.clone(ind) for ind in offspring]
        
        # Crossover
        for i in range(1, len(offspring), 2):
            if random.random() < 0.7:
                offspring[i-1], offspring[i] = toolbox.mate(offspring[i-1], offspring[i])
                del offspring[i-1].fitness.values
                del offspring[i].fitness.values
        
        # Mutation
        for mutant in offspring:
            if random.random() < 0.2:
                toolbox.mutate(mutant)
                del mutant.fitness.values
        
        # Baldwinian evaluation
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            base_fitness = toolbox.evaluate(ind)[0]
            
            if random.random() < local_search_prob:
                improved, n_evals = bit_flip_hill_climb(ind, toolbox.evaluate, local_search_iters)
                total_ls_evals += n_evals
                learned_fitness = toolbox.evaluate(improved)[0]
                # Use learned fitness, keep original genome
                ind.fitness.values = (learned_fitness,)
            else:
                ind.fitness.values = (base_fitness,)
        
        population[:] = offspring
        
        record = stats.compile(population)
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        
        if (gen + 1) % 10 == 0:
            print(f"Gen {gen+1}: Best={record['max']:.4f}, Avg={record['avg']:.4f}")
    
    return population, logbook, total_ls_evals

print("Baldwinian GA function defined.")

## 5. Experimental Comparison

### Key Concept: Compare pure GA, Lamarckian, and Baldwinian approaches.

In [ ]:
# Purpose: Setup common toolbox for fair comparison
# Key Concept: Same parameters except local search strategy

# Create toolbox
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=N_FEATURES
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_features, X_data=X_train_scaled, y_data=y_train)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

print("Common toolbox configured for experiments.")

### 5.1 Run Pure GA (Baseline)

In [ ]:
# Purpose: Run standard GA as baseline
# Key Concept: No local search, pure evolutionary approach

print("Running Pure GA (Baseline)...\n")

stats_pure = tools.Statistics(key=lambda ind: ind.fitness.values)
stats_pure.register("avg", np.mean)
stats_pure.register("max", np.max)

pop_pure = toolbox.population(n=50)

pop_pure, log_pure = algorithms.eaSimple(
    pop_pure,
    toolbox,
    cxpb=0.7,
    mutpb=0.2,
    ngen=30,
    stats=stats_pure,
    verbose=False
)

best_pure = tools.selBest(pop_pure, 1)[0]
print(f"\nPure GA Results:")
print(f"  Best fitness: {best_pure.fitness.values[0]:.4f}")
print(f"  Features: {sum(best_pure)}")

### 5.2 Run Lamarckian GA

In [ ]:
# Purpose: Run Lamarckian hybrid GA
# Key Concept: Local search modifies genome

print("\nRunning Lamarckian GA...\n")

pop_lamarck, log_lamarck, ls_evals_lamarck = lamarckian_ga(
    toolbox,
    population_size=50,
    n_generations=30,
    local_search_prob=0.3,
    local_search_iters=3
)

best_lamarck = tools.selBest(pop_lamarck, 1)[0]
print(f"\nLamarckian GA Results:")
print(f"  Best fitness: {best_lamarck.fitness.values[0]:.4f}")
print(f"  Features: {sum(best_lamarck)}")
print(f"  Local search evaluations: {ls_evals_lamarck}")

### 5.3 Run Baldwinian GA

In [ ]:
# Purpose: Run Baldwinian hybrid GA
# Key Concept: Local search affects fitness only

print("\nRunning Baldwinian GA...\n")

pop_baldwin, log_baldwin, ls_evals_baldwin = baldwinian_ga(
    toolbox,
    population_size=50,
    n_generations=30,
    local_search_prob=0.3,
    local_search_iters=3
)

best_baldwin = tools.selBest(pop_baldwin, 1)[0]
print(f"\nBaldwinian GA Results:")
print(f"  Best fitness: {best_baldwin.fitness.values[0]:.4f}")
print(f"  Features: {sum(best_baldwin)}")
print(f"  Local search evaluations: {ls_evals_baldwin}")

### 5.4 Compare Results

In [ ]:
# Purpose: Comprehensive comparison of all methods
# Key Concept: Analyze convergence, final quality, efficiency

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extract data
gen_pure = log_pure.select("gen")
max_pure = log_pure.select("max")

gen_lamarck = log_lamarck.select("gen")
max_lamarck = log_lamarck.select("max")

gen_baldwin = log_baldwin.select("gen")
max_baldwin = log_baldwin.select("max")

# Plot 1: Convergence curves
axes[0].plot(gen_pure, max_pure, 'b-o', linewidth=2, markersize=4, label='Pure GA')
axes[0].plot(gen_lamarck, max_lamarck, 'r-s', linewidth=2, markersize=4, label='Lamarckian')
axes[0].plot(gen_baldwin, max_baldwin, 'g-^', linewidth=2, markersize=4, label='Baldwinian')
axes[0].set_xlabel('Generation', fontsize=12)
axes[0].set_ylabel('Best Fitness', fontsize=12)
axes[0].set_title('Convergence Comparison', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Plot 2: Summary comparison
methods = ['Pure GA', 'Lamarckian', 'Baldwinian']
best_fitness = [
    best_pure.fitness.values[0],
    best_lamarck.fitness.values[0],
    best_baldwin.fitness.values[0]
]
n_features = [
    sum(best_pure),
    sum(best_lamarck),
    sum(best_baldwin)
]

x = np.arange(len(methods))
width = 0.35

bars1 = axes[1].bar(x - width/2, best_fitness, width, label='Best Fitness', color='steelblue')
ax2 = axes[1].twinx()
bars2 = ax2.bar(x + width/2, n_features, width, label='Num Features', color='coral')

axes[1].set_xlabel('Method', fontsize=12)
axes[1].set_ylabel('Best Fitness', fontsize=12, color='steelblue')
ax2.set_ylabel('Number of Features', fontsize=12, color='coral')
axes[1].set_title('Final Performance', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='coral')

# Combined legend
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

# Summary table
print("\nComparison Summary:")
print("="*70)
print(f"{'Method':<15} {'Best Fitness':>12} {'Features':>10} {'LS Evals':>12}")
print("-"*70)
print(f"{'Pure GA':<15} {best_pure.fitness.values[0]:>12.4f} {sum(best_pure):>10} {0:>12}")
print(f"{'Lamarckian':<15} {best_lamarck.fitness.values[0]:>12.4f} {sum(best_lamarck):>10} {ls_evals_lamarck:>12}")
print(f"{'Baldwinian':<15} {best_baldwin.fitness.values[0]:>12.4f} {sum(best_baldwin):>10} {ls_evals_baldwin:>12}")

## 6. Exercises

### Exercise 6.1: Adaptive Local Search Frequency

**Task**: Implement adaptive local search where probability starts high (exploration) and decreases over generations (exploitation).

**Expected Output**: Comparison with fixed local search probability.

<details>
<summary>Hint</summary>
Use `local_search_prob = max_prob * (1 - gen / max_gen)`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 6.2: Multi-Start Local Search

**Task**: Instead of one local search run, try multiple random restarts from an individual and keep the best. Compare efficiency.

**Expected Output**: Analysis of whether multiple restarts improve solution quality.

<details>
<summary>Hint</summary>
Perturb individual slightly, run local search from each perturbation, keep best result.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 6.3: Hybrid with Simulated Annealing

**Task**: Replace hill climbing with simulated annealing as local search. Implement temperature schedule and acceptance probability.

**Expected Output**: Comparison of hill climbing vs simulated annealing local search.

<details>
<summary>Hint</summary>
Accept worse solutions with probability `exp(-(new_fitness - old_fitness) / temperature)`. Decrease temperature over iterations.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

## 7. Summary

### Key Takeaways

1. **Hybrid Advantage**: Combines global search with local refinement
2. **Lamarckian**: Local improvements inherited (faster convergence)
3. **Baldwinian**: Rewards learning potential without changing genome
4. **Trade-offs**: Hybrid methods use more evaluations but find better solutions
5. **Memetic Algorithms**: Cultural evolution + biological evolution
6. **Application-Specific**: Choose local search method based on problem structure

### Lamarckian vs Baldwinian

| Aspect | Lamarckian | Baldwinian |
|--------|-----------|------------|
| **Genome modification** | Yes | No |
| **Inheritance** | Direct | Indirect |
| **Convergence speed** | Faster | Slower |
| **Diversity** | Lower | Higher |
| **Robustness** | Lower | Higher |
| **Best for** | Exploitation | Exploration |

### Local Search Design Principles

1. **Computational budget**: Balance GA and local search evaluations
2. **Search neighborhood**: Small (bit flips) vs large (greedy)
3. **Application frequency**: Every individual vs selected individuals
4. **Iteration limit**: Prevent excessive local search time
5. **Adaptive strategies**: Adjust based on evolution stage

### Performance Considerations

**Hybrid methods work best when:**
- Solution landscape has local structure
- Fitness evaluation is not too expensive
- Quality matters more than speed
- Problem has clear local optima

**Pure GA is better when:**
- Fitness evaluation is very expensive
- Solution space is highly rugged
- Diversity is critical
- Computational budget is tight

### Practical Recommendations

1. **Start simple**: Try pure GA first as baseline
2. **Experiment**: Test different local search probabilities (0.1-0.5)
3. **Monitor**: Track local search contribution to overall improvement
4. **Adapt**: Use Lamarckian for exploitation, Baldwinian for exploration
5. **Balance**: Allocate ~30% of evaluations to local search

### What's Next

- **Advanced hybrids**: Variable neighborhood search, iterated local search
- **Parallel hybrids**: Multiple populations with local search
- **Machine learning integration**: Learn when to apply local search
- **Real-world deployment**: Apply to large-scale feature selection

### Additional Resources

- **Memetic Algorithms**: Moscato & Cotta (2003) - "A Gentle Introduction to Memetic Algorithms"
- **Lamarckian vs Baldwinian**: Whitley et al. (1994) - "Lamarckian Evolution, The Baldwin Effect and Function Optimization"
- **Hybrid Metaheuristics**: Talbi (2009) - "Metaheuristics: From Design to Implementation"